# AlgoArena: Sharpe-first model, version 3

This notebook uses the competition files already committed in HRT-Game. It generates local audit diagnostics and a submission CSV without using hidden labels.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/content/Quant-Comp-2026')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ABCPeter123/Quant-Comp-2026.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', str(repo / 'algoarena' / 'requirements.txt')], check=True)


In [ ]:
data_dir = repo / 'algoarena'
subprocess.run([sys.executable, str(data_dir / 'algoarena_sharpe_v3.py'), '--data-dir', str(data_dir), '--check-data'], check=True)
print('Competition files:', data_dir)


## Validate, audit, and generate predictions

Three index-ordered validation folds choose a Sharpe-first configuration. The sealed 20% audit is held aside until selection. Reports are saved in a unique experiment folder.

In [ ]:
from datetime import datetime, timezone
run_name = datetime.now(timezone.utc).strftime('colab_%Y%m%dT%H%M%SZ')
report_dir = data_dir / 'runs' / run_name
subprocess.run([sys.executable, str(data_dir / 'algoarena_sharpe_v3.py'),
                '--data-dir', str(data_dir), '--output-dir', str(report_dir)], check=True)
print('Reports:', report_dir)
print('Submission:', data_dir / 'predictions.csv')

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
report = json.loads((report_dir / 'report.json').read_text())
print('Validation Sharpe:', round(report['selection']['oof_sharpe'], 3))
print('Sealed audit Sharpe:', round(report['audit']['sharpe'], 3))
print('Audit bootstrap CI:', report['audit']['sharpe_bootstrap_95pct_ci'])
print('Baseline Sharpes:', {k: round(v['sharpe'], 3) for k,v in report['baselines'].items()})
display(pd.read_csv(report_dir / 'candidate_scores.csv').head(10))
display(pd.read_csv(report_dir / 'audit_breakdowns.csv'))
display(Image(filename=str(report_dir / 'audit_equity_curve.png')))

In [ ]:
import shutil
archive = shutil.make_archive(str(report_dir), 'zip', root_dir=report_dir)
print('Submission:', data_dir / 'predictions.csv')
print('Debug bundle:', archive)
try:
    from google.colab import files
    files.download(str(data_dir / 'predictions.csv'))
    files.download(archive)
except ImportError:
    pass